In [13]:
import pandas as pd
import numpy as np

# Read in the csv
zillow_df = pd.read_csv("Resources/Neighborhood_zillow_real_estate_data.csv")

In [14]:
# Pull only the data for Denver
denver_zillow_df = zillow_df.query("City == 'Denver'")

# Drop Columns not needed
denver_zillow_df = denver_zillow_df.drop(columns=["RegionID", "SizeRank", "RegionType", "StateName", "State", "Metro", "CountyName", "City"], axis=1)

#Rename Region Name Column
denver_zillow_df.rename(columns={"RegionName":"Neighborhood"}, inplace=True)
denver_zillow_df.head()

,Neighborhood,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,2000-06-30,2000-07-31,2000-08-31,2000-09-30,...,2024-05-31,2024-06-30,2024-07-31,2024-08-31,2024-09-30,2024-10-31,2024-11-30,2024-12-31,2025-01-31,2025-02-28
235,Gateway - Green Valley Ranch,159100.744442,159765.325210,160535.294508,162451.362079,164505.655223,166766.520599,168759.025196,170778.308031,172897.754144,...,491760.579762,490858.062361,489745.958237,488726.750030,487995.892604,487302.918410,486903.731696,486630.545212,485837.941119,484004.684275
329,Montbello,126411.509704,127119.793823,127826.964815,129558.027973,131420.965731,133511.706505,135523.103627,137493.170285,139465.789053,...,441051.882758,440704.584839,439855.248083,439006.425223,438219.703796,437274.333697,436726.770604,436244.288979,435538.427024,433816.969981
410,Central Park,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,810438.023097,808244.803889,805155.863258,802598.736735,801195.640708,799657.640124,798600.457230,798423.875011,797461.775214,795133.205809
602,Hampden,166128.053070,166995.838447,168043.302926,170454.810786,173004.765599,175590.495277,177909.646389,180145.891194,182401.089311,...,525244.032565,523673.717091,522134.523951,521460.718137,521413.062907,521744.191939,522074.783631,522563.712560,522277.831180,520475.767806
681,Five Points,188674.759005,190474.159808,191852.746828,195655.380983,199917.399170,204490.844183,209338.902439,214221.458758,218864.694301,...,618676.421526,615327.394116,611031.871498,607721.730126,605831.104045,604129.703444,603280.732065,603091.967420,603290.808745,603176.667680


In [15]:
# Melt it into long format
denver_zillow_df_long = denver_zillow_df.melt(id_vars=["Neighborhood"], var_name="Date", value_name="Price")

# Convert 'Date' to datetime
denver_zillow_df_long['Date'] = pd.to_datetime(denver_zillow_df_long['Date'])
denver_zillow_df_long.head()

,Neighborhood,Date,Price
0,Gateway - Green Valley Ranch,2000-01-31,159100.744442
1,Montbello,2000-01-31,126411.509704
2,Central Park,2000-01-31,NaN
3,Hampden,2000-01-31,166128.053070
4,Five Points,2000-01-31,188674.759005


In [16]:
# Add time features
denver_zillow_df_long['Year'] = denver_zillow_df_long['Date'].dt.year
denver_zillow_df_long['Month'] = denver_zillow_df_long['Date'].dt.month
denver_zillow_df_long['TimeIndex'] = (denver_zillow_df_long['Year'] - denver_zillow_df_long['Year'].min()) * 12 + denver_zillow_df_long['Month']

# Encode the 'Month' feature as cyclical variables using sine and cosine.
# This preserves the cyclical nature of months (e.g., December and January are close together).
# Helps the machine learning model learn seasonal patterns more effectively.
denver_zillow_df_long['Month_sin'] = np.sin(2 * np.pi * denver_zillow_df_long['Month'] / 12)
denver_zillow_df_long['Month_cos'] = np.cos(2 * np.pi * denver_zillow_df_long['Month'] / 12)

In [17]:
# Check how many rows and columns we have in the dataset
print(denver_zillow_df_long.shape)

# Check for null values
print(denver_zillow_df_long.isnull().sum())

(22952, 8)
Neighborhood      0
Date              0
Price           550
Year              0
Month             0
TimeIndex         0
Month_sin         0
Month_cos         0
dtype: int64


In [18]:
# Remove null values in the "Price" column
denver_zillow_df_long = denver_zillow_df_long.dropna(subset=['Price'])

# Check that the null values have been removed
print(denver_zillow_df_long.isnull().sum())

Neighborhood    0
Date            0
Price           0
Year            0
Month           0
TimeIndex       0
Month_sin       0
Month_cos       0
dtype: int64


In [19]:
#Export CSV
denver_zillow_df_long.to_csv("Resources/zillow_cleaned.csv")